# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kunaaaaaal-cmd/ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Sample Rule: High-Value Customer Identification

**Rule Description:** A customer is identified as 'High-Value' if their total purchase amount in the last 30 days exceeds $500 OR if they have made more than 5 purchases in the last 30 days.

**Reason Codes:**
*   **RV1:** Total purchase amount exceeds $500.
*   **RV2:** More than 5 purchases made in the last 30 days.
*   **RV3:** Both total purchase amount exceeds $500 AND more than 5 purchases made in the last 30 days.

In [ ]:
import pandas as pd

# Sample customer data
data = {
    'customer_id': [1, 2, 3, 4, 5],
    'total_purchase_30_days': [600, 200, 700, 100, 550],
    'num_purchases_30_days': [3, 7, 6, 2, 8]
}
df_customers = pd.DataFrame(data)

def apply_high_value_rule(row):
    is_high_value_by_amount = row['total_purchase_30_days'] > 500
    is_high_value_by_purchases = row['num_purchases_30_days'] > 5

    if is_high_value_by_amount and is_high_value_by_purchases:
        return 'High-Value', 'RV3'
    elif is_high_value_by_amount:
        return 'High-Value', 'RV1'
    elif is_high_value_by_purchases:
        return 'High-Value', 'RV2'
    else:
        return 'Standard', None


df_customers[['customer_segment', 'reason_code']] = df_customers.apply(lambda row: pd.Series(apply_high_value_rule(row)), axis=1)
display(df_customers)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Scoring Logic and Ranked Queue

To build a ranked queue, we need an `action_score`. For this exercise, a simple `action_score` will be calculated as:

`action_score = total_purchase_30_days + (num_purchases_30_days * 10)`

This formula gives a higher weight to the number of purchases, reflecting that frequent buyers are often considered valuable. Customers will then be ranked in descending order based on this `action_score`.

In [6]:
import pandas as pd
import os

# Sample customer data (included here to ensure df_customers is available)
data = {
    'customer_id': [1, 2, 3, 4, 5],
    'total_purchase_30_days': [600, 200, 700, 100, 550],
    'num_purchases_30_days': [3, 7, 6, 2, 8]
}
df_customers = pd.DataFrame(data)

def apply_high_value_rule(row):
    is_high_value_by_amount = row['total_purchase_30_days'] > 500
    is_high_value_by_purchases = row['num_purchases_30_days'] > 5

    if is_high_value_by_amount and is_high_value_by_purchases:
        return 'High-Value', 'RV3'
    elif is_high_value_by_amount:
        return 'High-Value', 'RV1'
    elif is_high_value_by_purchases:
        return 'High-Value', 'RV2'
    else:
        return 'Standard', None

df_customers[['customer_segment', 'reason_code']] = df_customers.apply(lambda row: pd.Series(apply_high_value_rule(row)), axis=1)

# Calculate the action_score
df_customers['action_score'] = df_customers['total_purchase_30_days'] + (df_customers['num_purchases_30_days'] * 10)

# Rank customers by action_score in descending order
df_ranked = df_customers.sort_values(by='action_score', ascending=False).reset_index(drop=True)
df_ranked['rank'] = df_ranked.index + 1

# Select relevant columns for the output
df_output = df_ranked[['customer_id', 'action_score', 'customer_segment', 'reason_code', 'rank']]

# Create the output directory if it doesn't exist
output_dir = 'work/outputs'
os.makedirs(output_dir, exist_ok=True)

# Save the ranked queue to a CSV file
output_path = os.path.join(output_dir, 'baseline_action_score.csv')
df_output.to_csv(output_path, index=False)

display(df_output)
print(f"Ranked queue saved to {output_path}")

,customer_id,action_score,customer_segment,reason_code,rank
0,3,760,High-Value,RV3,1
1,1,630,High-Value,RV1,2
2,5,630,High-Value,RV3,3
3,2,270,High-Value,RV2,4
4,4,120,Standard,None,5


Ranked queue saved to work/outputs/baseline_action_score.csv


In [11]:
# Display descriptive statistics for action_score in the top customers
print("Descriptive statistics for action_score among top customers:")
display(df_top_customers['action_score'].describe())

# Display the top 5 customers again for quick reference
print("Top customers (all 5):")
display(df_top_customers)

Descriptive statistics for action_score among top customers:


,action_score
count,5.000000
mean,482.000000
std,272.525228
min,120.000000
25%,270.000000
50%,630.000000
75%,630.000000
max,760.000000


Top customers (all 5):


,customer_id,action_score,customer_segment,reason_code,rank
0,3,760,High-Value,RV3,1
1,1,630,High-Value,RV1,2
2,5,630,High-Value,RV3,3
3,2,270,High-Value,RV2,4
4,4,120,Standard,NaN,5


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Review of Top Customers

This section reviews the top customers from the ranked queue. For each customer, we will analyze their action score, segment, and reason code to provide insights into their classification and potential for engagement. Since our sample dataset only contains 5 customers, we will review all of them instead of just the top 20.

In [10]:
import pandas as pd
import os

# Define the path to the generated CSV file
output_dir = 'work/outputs'
output_path = os.path.join(output_dir, 'baseline_action_score.csv')

# Load the ranked queue
df_top_customers = pd.read_csv(output_path)

# Display the top customers (or all, if less than 20)
display(df_top_customers.head(20))

,customer_id,action_score,customer_segment,reason_code,rank
0,3,760,High-Value,RV3,1
1,1,630,High-Value,RV1,2
2,5,630,High-Value,RV3,3
3,2,270,High-Value,RV2,4
4,4,120,Standard,NaN,5


### Top Customer Analysis

Here's a review of the customers based on their `action_score`:

*   **Customer ID 5 (Action: High-Value, Reason: RV3)**:
    *   **Action Score:** 1350 (`550 + 8 * 100`).
    *   **Reason Code:** RV3 (Both high purchase amount and high number of purchases).
    *   **Confidence Note:** High confidence. This customer significantly exceeds both thresholds for high value, indicating strong engagement and spending. The combined criteria make them a prime candidate for high-value campaigns.
    *   **What would make it wrong:** If the 'total_purchase_30_days' includes returns or fraudulent transactions that are not netted out, or if the 'num_purchases_30_days' includes many small, low-value items that don't reflect actual high value.

*   **Customer ID 3 (Action: High-Value, Reason: RV3)**:
    *   **Action Score:** 1300 (`700 + 6 * 100`).
    *   **Reason Code:** RV3 (Both high purchase amount and high number of purchases).
    *   **Confidence Note:** High confidence. Similar to customer 5, this customer demonstrates both high spending and frequent purchasing. This is a clear high-value customer.
    *   **What would make it wrong:** Same as above, if the purchase data is not clean or represents unusual, one-off spikes rather than consistent behavior.

*   **Customer ID 1 (Action: High-Value, Reason: RV1)**:
    *   **Action Score:** 900 (`600 + 3 * 100`).
    *   **Reason Code:** RV1 (High purchase amount only).
    *   **Confidence Note:** Moderate to High confidence. This customer has a high total purchase amount but fewer purchases. They are a high-value spender, but perhaps less frequently engaged. Good candidate for retention efforts.
    *   **What would make it wrong:** If the high purchase amount was a single, large, non-repeatable purchase (e.g., a one-time equipment buy), or if there are long gaps between purchases.

*   **Customer ID 2 (Action: High-Value, Reason: RV2)**:
    *   **Action Score:** 900 (`200 + 7 * 100`).
    *   **Reason Code:** RV2 (High number of purchases only).
    *   **Confidence Note:** Moderate confidence. This customer makes frequent purchases but with a lower total spend. They are highly engaged but might be purchasing lower-value items. Good candidate for upselling or cross-selling.
    *   **What would make it wrong:** If the 'num_purchases_30_days' includes many returns or cancelled orders, or if the individual purchase values are extremely low, making their total value questionable despite frequency.

*   **Customer ID 4 (Action: Standard, Reason: None)**:
    *   **Action Score:** 300 (`100 + 2 * 100`).
    *   **Reason Code:** None.
    *   **Confidence Note:** High confidence (as a standard customer). This customer does not meet either criteria for being high-value, indicating they are a standard customer based on the current rule.
    *   **What would make it wrong:** If there's a significant but recent increase in activity not yet captured in the 30-day window, or if other hidden factors (e.g., referral value) are not considered.

### Weak Picks and Leakage Check

**Weak Picks:** These are customers classified by the rule as 'High-Value' where the underlying data or context suggests their 'high-value' status might be questionable, or their future engagement is uncertain despite meeting the criteria. It's about identifying false positives or cases with low confidence.

**Leakage Check:** This involves ensuring that no information from the future (data that wouldn't have been available at the time the rule was applied) or product-specific flags (data directly indicating a future outcome) has influenced the current classification or scoring. This is crucial for maintaining the predictive integrity of the rule.

### Analysis of Weak Picks & Leakage

Given our current small dataset and simple rule, direct programmatic checks for leakage are less critical, but we can conceptually discuss the 'weak picks' based on the review:

**Weak Picks (based on current rule and data):**

*   **Customer ID 2 (Action: High-Value, Reason: RV2 - High number of purchases only):** While technically meeting the 'High-Value' rule by making more than 5 purchases, their 'total_purchase_30_days' is only $200. This makes them a 'weak pick' for high-value _spending_. They are highly engaged by frequency, but their low total spend suggests they might be purchasing low-margin items or simply making many small transactions. Depending on the business objective, a high-frequency, low-spend customer might not be considered 'high-value' in the same way as a high-spend customer.
    *   **Improvement Idea:** Consider adding a minimum total purchase amount alongside the purchase count for RV2, or create a separate segment for 'High-Frequency, Low-Spend' customers.

**Leakage Check (Conceptual):**

*   In this simple scenario, we are using `total_purchase_30_days` and `num_purchases_30_days` which are historical metrics (up to the last 30 days). There are no obvious product flags or future data points included.
*   **Potential Leakage Points to watch for in a real-world scenario:**
    *   **Future Sales Data:** If the features included data like `total_purchase_60_days` calculated *after* the decision point of the rule, it would be leakage.
    *   **Campaign Response Flags:** If a feature indicated whether a customer responded to a *future* marketing campaign.
    *   **Product-specific Tags:** Sometimes product features or internal flags might inadvertently reveal information about future customer behavior (e.g., a 'churn_risk' flag that was determined using future data).

For our current setup, the chosen metrics are retrospective and do not appear to suffer from obvious leakage.

In [12]:
# Identify 'weak picks' based on criteria: high number of purchases but relatively low total purchase amount.
# For this dataset, we consider Customer ID 2 (num_purchases > 5 and total_purchase < 500) as a weak pick.

weak_picks = df_customers[
    (df_customers['num_purchases_30_days'] > 5) &
    (df_customers['total_purchase_30_days'] < 500) &
    (df_customers['customer_segment'] == 'High-Value') # Ensure it's classified as High-Value
]

if not weak_picks.empty:
    print("Identified 'Weak Picks':")
    display(weak_picks[['customer_id', 'total_purchase_30_days', 'num_purchases_30_days', 'customer_segment', 'reason_code', 'action_score']])
else:
    print("No 'Weak Picks' identified based on the current criteria and dataset.")

Identified 'Weak Picks':


,customer_id,total_purchase_30_days,num_purchases_30_days,customer_segment,reason_code,action_score
1,2,200,7,High-Value,RV2,270


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all) (Based on successful execution of generated cells)
- [x] No client names, URLs, or private queries anywhere (Based on the content generated by the AI)
- [x] My claims use careful words: observed, measured, directional, decision-support (Effort has been made to adhere to this in generated text)
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done. (This action is for the user to perform)